In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


def create_visualizer():
    """Return a ModelVisualizer with the P2 experimental notice acknowledged.

    kailash-ml 2.2.2 emits ExperimentalWarning (a UserWarning) at
    ModelVisualizer construction; exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly here — the ONE construction site
    every course file should use.
    """
    import warnings as _warnings

    from kailash_ml import ModelVisualizer
    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()

# ── SDK-internal deprecation nag filter (strict-gate hygiene) ────────────
# torch 2.12's ONNX exporter warns that dynamic_axes is "not recommended"
# under dynamo=True — a deprecation notice inside OnnxBridge's internals,
# not a correctness signal, and fatal under warnings-as-errors. Acknowledged
# here (kailash_helpers is imported by every shared.mlfpNN package, so every
# technique file gets it); message-matched, everything else still errors.
import warnings as _warnings

_warnings.filterwarnings(
    "ignore", message=r".*dynamic_axes.*not recommended.*", category=UserWarning
)
# matplotlib's Agg backend (forced in headless/script runs) nags
# "FigureCanvasAgg is non-interactive, and thus cannot be shown" on every
# plt.show() — a backend notice, not a correctness signal.
_warnings.filterwarnings(
    "ignore", message=r".*non-interactive.*cannot be shown.*", category=UserWarning
)



# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_4.py ──
"""
Shared infrastructure for MLFP03 Exercise 4 — Gradient Boosting Deep Dive.

Contains: Singapore credit-scoring data loader, preprocessing pipeline
wrapper, numpy/polars conversion, model-zoo factories (XGBoost/LightGBM/
CatBoost with identical defaults), evaluation metric helpers, and the
output directory used by every technique file.

Technique-specific code (from-scratch boosting loops, split-gain formulas,
hyperparameter sweeps, early-stopping analysis) does NOT belong here — it
lives in the per-technique files under `modules/mlfp03/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONFIG — output directory, random seeds, dataset tag
# ════════════════════════════════════════════════════════════════════════

SEED = 42
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer.
# ``screen_single_feature_leakage`` below is the EDA check that exposes it.
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")

OUTPUT_DIR = Path("outputs") / "mlfp03" / "ex_4_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit-scoring, shared across all four techniques
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> pl.DataFrame:
    """Load the Singapore credit-scoring dataset via MLFPDataLoader."""
    loader = MLFPDataLoader()
    return loader.load(DATASET_MODULE, DATASET_FILE)


def screen_single_feature_leakage(
    df: pl.DataFrame,
    target: str = TARGET_COLUMN,
    *,
    flag_auc: float = 0.95,
) -> pl.DataFrame:
    """EDA leakage screen: how well does EACH numeric column alone rank the target?

    A legitimate credit feature on its own rarely exceeds AUC ~0.75. A column
    that separates defaulters almost perfectly by itself is almost always
    information recorded after the outcome (or the outcome in disguise).

    Returns a polars DataFrame (feature, single_feature_auc, suspicious)
    sorted by AUC, where AUC is orientation-free (max(auc, 1 - auc)).
    """
    y = df[target].to_numpy()
    rows: list[dict[str, Any]] = []
    for col, dtype in df.schema.items():
        if col == target or not dtype.is_numeric():
            continue
        x = df[col].fill_null(df[col].median()).to_numpy().astype(float)
        if np.unique(x).size < 2:
            continue
        auc = float(roc_auc_score(y, x))
        auc = max(auc, 1.0 - auc)
        rows.append(
            {"feature": col, "single_feature_auc": auc, "suspicious": auc >= flag_auc}
        )
    return pl.DataFrame(rows).sort("single_feature_auc", descending=True)


def prepare_credit_split() -> dict[str, Any]:
    """Load credit data and return a train/test split ready for boosting.

    Returns a dict with:
      X_train, y_train, X_test, y_test : numpy arrays
      feature_names                    : list[str]
      default_rate                     : float (positive-class prevalence)

    ``CREDIT_NON_FEATURE_COLUMNS`` (the row ID and the post-outcome leak
    column) are dropped BEFORE preprocessing so no model in this exercise
    can see them.

    Tree models do not need normalisation, so we set ``normalize=False``.
    Categoricals are ordinal-encoded because XGBoost/LightGBM expect numeric
    input; CatBoost would accept raw categoricals but we keep the pipeline
    consistent across all three libraries for a fair comparison.
    """
    credit = load_credit_data().drop(CREDIT_NON_FEATURE_COLUMNS)

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target=TARGET_COLUMN,
        test_size=0.2,
        seed=SEED,
        normalize=False,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(credit[TARGET_COLUMN].mean()),
    }


def categorical_feature_indices(feature_names: list[str]) -> list[int]:
    """Column indices (in ``feature_names``) of the dataset's string categoricals.

    ``prepare_credit_split`` ordinal-encodes these (gender, race, region,
    loan_purpose, ...). CatBoost can instead treat them as true categories.
    """
    schema = load_credit_data().schema
    return [
        i for i, name in enumerate(feature_names) if schema.get(name) == pl.String
    ]


def as_catboost_categoricals(X: np.ndarray, cat_idx: list[int]) -> np.ndarray:
    """Return an object array where the categorical columns hold integer codes.

    CatBoost refuses float-valued categorical columns; integer codes (or
    strings) are accepted and treated as unordered categories, so the
    arbitrary ordinal order no longer matters.
    """
    out = X.astype(object)
    for i in cat_idx:
        out[:, i] = X[:, i].astype(int)
    return out


# ════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES — identical defaults for fair comparison
# ════════════════════════════════════════════════════════════════════════


def make_xgboost(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build an XGBoost classifier with course-standard defaults."""
    import xgboost as xgb

    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0,
        **extra,
    )


def make_lightgbm(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build a LightGBM classifier with course-standard defaults."""
    import lightgbm as lgb

    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=31,
        random_state=SEED,
        verbose=-1,
        **extra,
    )


def make_catboost(
    iterations: int = 500,
    learning_rate: float = 0.1,
    depth: int = 6,
    **extra: Any,
):
    """Build a CatBoost classifier with course-standard defaults."""
    import catboost as cb

    return cb.CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=SEED,
        verbose=0,
        **extra,
    )


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — shared metric helpers for imbalanced binary classification
# ════════════════════════════════════════════════════════════════════════


def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Return the full boosting-eval metric bundle.

    AUC-PR is the primary metric — with a 12% default rate, AUC-ROC rewards
    models that rank negatives correctly even when they miss every default.
    """
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def print_metrics(
    name: str, metrics: dict[str, float], train_time: float | None = None
) -> None:
    """One-line headline: AUC-ROC | AUC-PR | Log Loss | F1 | Time."""
    time_str = f" | time={train_time:.2f}s" if train_time is not None else ""
    print(
        f"  {name}: "
        f"AUC-ROC={metrics['auc_roc']:.4f} | "
        f"AUC-PR={metrics['auc_pr']:.4f} | "
        f"log_loss={metrics['log_loss']:.4f} | "
        f"F1={metrics['f1']:.4f}"
        f"{time_str}"
    )


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH GRADIENT BOOSTING (used by 01_boosting_theory.py)
# ════════════════════════════════════════════════════════════════════════


def xgb_split_gain(
    g_left: float,
    h_left: float,
    g_right: float,
    h_right: float,
    lambda_reg: float = 1.0,
    gamma: float = 0.0,
) -> float:
    """XGBoost split-gain formula from 2nd-order Taylor expansion of log-loss.

    Gain = ½ [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ
    """
    left_score = g_left**2 / (h_left + lambda_reg)
    right_score = g_right**2 / (h_right + lambda_reg)
    parent_score = (g_left + g_right) ** 2 / (h_left + h_right + lambda_reg)
    return 0.5 * (left_score + right_score - parent_score) - gamma


def make_1d_demo(n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate a 1D logistic-shaped binary classification demo.

    Used by the from-scratch boosting loop to show that residual-fitting
    converges in just a handful of rounds.
    """
    rng = np.random.default_rng(SEED)
    x = rng.uniform(0, 1, n).reshape(-1, 1)
    true_proba = 1 / (1 + np.exp(-10 * (x.ravel() - 0.5)))
    y = rng.binomial(1, true_proba)
    return x, y




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 4.1: Boosting Theory (From-Scratch + XGBoost Split Gain)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Explain why boosting reduces BIAS while bagging reduces VARIANCE
#   - Run AdaBoost by hand: re-weight the rows the last stump got wrong
#   - Implement gradient boosting from scratch with shallow decision trees
#   - Derive the XGBoost split-gain formula from a 2nd-order Taylor
#     expansion of the log-loss
#   - Interpret λ (leaf-weight L2) and γ (min split loss) as structural
#     regularisers on trees
#   - Explain when a bank's credit team would refuse a split even if it
#     "looks" informative (the pruning condition: Gain < γ)
#
# PREREQUISITES: Exercise 3 (decision trees, Random Forest). Boosting
# extends the same decision-tree primitive into a sequential ensemble.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — bias vs variance, sequential residual fitting
#   2. Build — AdaBoost warm-up, then a from-scratch gradient booster
#   3. Train — 10 rounds, watch residuals shrink; derive the split gain
#   4. Visualise — residual shrinkage + final probability surface (HTML)
#   5. Apply — an SME credit committee: when to refuse a split
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


load_dotenv()



## THEORY — Why Boosting Works


In [ ]:
# Bagging (Random Forest) trains many independent trees on bootstrap
# samples and averages them. This reduces VARIANCE because each tree sees
# a different slice of the data, but every tree makes the SAME kind of
# systematic mistakes as the others (high variance, same bias).
#
# Boosting is the opposite idea: train one tree, look at where it is
# WRONG, train the next tree to correct THAT mistake, repeat. This
# attacks BIAS directly — each new tree is fitted to the residuals
# (the negative gradient of the loss), so the ensemble's systematic
# error shrinks round by round.
#
# The additive-model form:
#     F_0(x) = log-odds of the positive class
#     F_m(x) = F_{m-1}(x) + η · h_m(x)
#
# where h_m is a shallow tree fitted to the pseudo-residuals
#     r_i = y_i - sigmoid(F_{m-1}(x_i))
#
# and η (learning rate) is a step-size that keeps each correction small
# enough that the ensemble does not overshoot. Smaller η → slower but
# more robust convergence. This is the fundamental mechanism behind
# XGBoost, LightGBM, and CatBoost — all three use the same recipe, they
# just differ in how they find the best split at each round.



## TASK 2a — BUILD: AdaBoost warm-up (boosting by RE-WEIGHTING rows)


In [ ]:
# AdaBoost (Freund & Schapire, 1997) was the first practical booster.
# Each round it fits a depth-1 tree ("stump") on WEIGHTED rows, then:
#     err_t   = Σ w_i over the rows the stump got wrong
#     α_t     = ½ ln((1 - err_t) / err_t)        (the stump's vote)
#     w_i    ← w_i · exp(+α_t) if wrong, w_i · exp(-α_t) if right; renormalise
# After the update the misclassified rows hold exactly HALF the total
# weight, so the next stump is forced to concentrate on them. Gradient
# boosting (below) generalises this: instead of re-weighting rows, each
# new tree is fit to the gradient of any differentiable loss.

print("\n" + "=" * 70)
print("  AdaBoost Warm-Up on a 1D Logistic Demo")
print("=" * 70)

x_demo, y_demo = make_1d_demo(n=200)
n_demo = len(y_demo)

w = np.full(n_demo, 1.0 / n_demo)
print(f"\n  {'Round':>6} {'weighted err':>13} {'alpha':>8} {'weight on missed rows':>22}")
print("  " + "─" * 54)
for t in range(1, 4):
    stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
    stump.fit(x_demo, y_demo, sample_weight=w)
    missed = stump.predict(x_demo) != y_demo
    err = float(w[missed].sum())
    alpha = 0.5 * np.log((1 - err) / err)
    w = w * np.exp(np.where(missed, alpha, -alpha))
    w = w / w.sum()
    missed_share = float(w[missed].sum())
    print(f"  {t:>6} {err:>13.4f} {alpha:>8.4f} {missed_share:>22.4f}")

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=SEED
)
ada.fit(x_demo, y_demo)
ada_acc = list(ada.staged_score(x_demo, y_demo))
print(
    f"\n  sklearn AdaBoost (50 stumps): training accuracy after 1 stump = "
    f"{ada_acc[0]:.4f}, after {len(ada_acc)} = {ada_acc[-1]:.4f}"
)



## TASK 2b — BUILD a from-scratch gradient booster


In [ ]:
print("\n" + "=" * 70)
print("  From-Scratch Gradient Boosting on 1D Logistic Demo")
print("=" * 70)

# Hyperparameters for the demo run
learning_rate = 0.3
n_rounds = 10

# F_0 = log-odds of the positive rate (baseline score)
pos_rate = y_demo.mean()
F = np.full(n_demo, np.log(pos_rate / (1 - pos_rate)))

print(f"\n  Initial F_0 = log-odds = {F[0]:.4f}")
print(f"  Initial sigmoid(F_0) = {1 / (1 + np.exp(-F[0])):.4f}")
print(f"\n  {'Round':>6} {'MSE(resid)':>14} {'Mean|resid|':>14} {'Accuracy':>10}")
print("  " + "─" * 48)

history = []  # collect per-round (round, mse, mean_abs, acc)
for m in range(1, n_rounds + 1):
    # Current probabilities
    p = 1 / (1 + np.exp(-F))

    # Pseudo-residuals = negative gradient of log-loss
    residuals = y_demo - p

    # Fit a shallow tree to the residuals (this is the boosting primitive)
    tree = DecisionTreeRegressor(max_depth=3, random_state=SEED)
    tree.fit(x_demo, residuals)
    h = tree.predict(x_demo)

    # Additive update with learning rate
    F = F + learning_rate * h

    # Metrics
    p_new = 1 / (1 + np.exp(-F))
    preds = (p_new >= 0.5).astype(int)
    acc = float((preds == y_demo).mean())
    mse_resid = float(np.mean(residuals**2))
    mean_abs_resid = float(np.mean(np.abs(residuals)))
    history.append((m, mse_resid, mean_abs_resid, acc))
    print(f"  {m:>6} {mse_resid:>14.6f} {mean_abs_resid:>14.6f} {acc:>10.4f}")

final_acc = history[-1][3]



### Checkpoint 1


In [ ]:
assert abs(missed_share - 0.5) < 1e-9, "AdaBoost puts half the weight on missed rows"
assert final_acc > 0.6, "From-scratch boosting should converge above 60% accuracy"
assert history[-1][1] < history[0][1], "MSE of residuals must shrink across rounds"
# INTERPRETATION: Every round, the MSE of the residuals decreases — the
# ensemble is learning the systematic error and subtracting it out. The
# learning rate 0.3 is aggressive; production boosters use 0.01-0.05 and
# compensate with more rounds.
print("\n[ok] Checkpoint 1 passed — from-scratch gradient boosting converged\n")



## TASK 3 — XGBoost Split-Gain Formula


In [ ]:
# XGBoost takes the additive model above and asks a sharper question: for
# a given split, how much does the loss decrease? It uses a second-order
# Taylor expansion:
#
#     L ≈ Σ [g_i · f(x_i) + ½ h_i · f(x_i)²] + Ω(f)
#
# where Ω(f) = γ·T + ½ λ Σ_j w_j² penalises a tree f with T leaves and
# leaf weights w_j.
# where g_i = ∂L/∂ŷ (first derivative) and h_i = ∂²L/∂ŷ² (second). For
# log-loss on a binary classification target:
#
#     g_i = p_i - y_i           (predicted minus actual)
#     h_i = p_i · (1 - p_i)     (prediction variance)
#
# Grouping the rows by the leaf j they fall in (G_j = Σ g_i, H_j = Σ h_i
# over that leaf), the loss is a quadratic in each w_j, minimised at
#
#     w_j* = - G_j / (H_j + λ)     with loss  - ½ G_j² / (H_j + λ) + γ
#
# Comparing the loss of one leaf (the parent) against two (the children),
# the split-gain formula falls out algebraically:
#
#     Gain = ½ [ G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ) ] - γ
#
# where G = Σ g_i, H = Σ h_i over each side. λ penalises large leaf
# weights (L2 regularisation on the tree's output), γ is a fixed cost of
# adding a leaf (pruning threshold). If Gain < 0 the split is refused.

print("\n" + "=" * 70)
print("  XGBoost Split-Gain Derivation — Worked Example")
print("=" * 70)

# Numerical example: a node with 100 defaults + 800 non-defaults
p_pred = 0.12  # current ensemble prediction (≈ population default rate)
g_default = p_pred - 1  # −0.88 per defaulter
g_no_default = p_pred - 0  # +0.12 per non-defaulter
h_per_sample = p_pred * (1 - p_pred)  # 0.1056

# Candidate split: left = 80 defaults + 50 non-defaults, right = the rest
g_left = 80 * g_default + 50 * g_no_default
h_left = 130 * h_per_sample
g_right = 20 * g_default + 750 * g_no_default
h_right = 770 * h_per_sample

gain = xgb_split_gain(g_left, h_left, g_right, h_right, lambda_reg=1.0, gamma=0.0)
print(f"\n  Split gain (λ=1, γ=0): {gain:.4f}")
print(f"    Left:  G_L={g_left:>8.2f}  H_L={h_left:>8.2f}")
print(f"    Right: G_R={g_right:>8.2f}  H_R={h_right:>8.2f}")

# Sensitivity: how λ and γ change the pruning decision
print("\n  --- Regularisation Effect on Split Gain ---")
print(f"  {'λ':>6} {'γ':>6} {'Gain':>10}  {'Decision':<15}")
print("  " + "─" * 44)
for lam in [0.0, 1.0, 10.0, 100.0]:
    for gam in [0.0, 1.0, 5.0]:
        g = xgb_split_gain(g_left, h_left, g_right, h_right, lam, gam)
        decision = "accept" if g > 0 else "PRUNE (Gain<0)"
        print(f"  {lam:>6.1f} {gam:>6.1f} {g:>10.4f}  {decision:<15}")



### Checkpoint 2


In [ ]:
assert gain > 0, "A well-separated split should have positive gain"
assert (
    xgb_split_gain(g_left, h_left, g_right, h_right, 100.0, 5.0) < gain
), "Heavy regularisation (λ=100, γ=5) should reduce the gain"
# INTERPRETATION: The gain formula is what makes XGBoost different from
# vanilla gradient boosting. λ limits how much any single leaf can swing
# the prediction; γ introduces a fixed cost for every new leaf. Together
# they force the tree to justify every split against the cost of
# complexity — this is structural regularisation, not just early stopping.
print("\n[ok] Checkpoint 2 passed — XGBoost split gain formula verified\n")



## TASK 4 — VISUALISE the residual shrinkage


In [ ]:
# The headline proof of boosting is that the mean absolute residual
# decreases every round. We plot it against the round index; a smooth
# curve shows the additive model is absorbing the signal.

fig = go.Figure()
rounds = [h[0] for h in history]
mean_abs = [h[2] for h in history]
acc_series = [h[3] for h in history]

fig.add_trace(
    go.Scatter(
        x=rounds,
        y=mean_abs,
        mode="lines+markers",
        name="Mean |residual|",
        yaxis="y1",
    )
)
fig.add_trace(
    go.Scatter(
        x=rounds,
        y=acc_series,
        mode="lines+markers",
        name="Accuracy",
        yaxis="y2",
    )
)
fig.update_layout(
    title="From-Scratch Gradient Boosting — Residual Shrinkage per Round",
    xaxis_title="Boosting round",
    yaxis=dict(title="Mean |residual|", side="left"),
    yaxis2=dict(title="Accuracy", overlaying="y", side="right", range=[0, 1]),
    legend=dict(x=0.02, y=0.98),
)
viz_path = OUTPUT_DIR / "ex4_01_residual_shrinkage.html"
fig.write_html(viz_path)
print(f"  Saved: {viz_path}")

# Decision-surface plot: probability vs x after final round
p_final = 1 / (1 + np.exp(-F))
order = np.argsort(x_demo.ravel())
surface = go.Figure()
surface.add_trace(
    go.Scatter(
        x=x_demo.ravel()[order],
        y=p_final[order],
        mode="lines",
        name="Booster P(y=1|x)",
    )
)
surface.add_trace(
    go.Scatter(
        x=x_demo.ravel(),
        y=y_demo,
        mode="markers",
        name="True labels",
        marker=dict(size=6, opacity=0.5),
    )
)
surface.update_layout(
    title="Boosted Probability Surface After 10 Rounds",
    xaxis_title="x",
    yaxis_title="P(default)",
)
surface_path = OUTPUT_DIR / "ex4_01_decision_surface.html"
surface.write_html(surface_path)
print(f"  Saved: {surface_path}")



## TASK 5 — APPLY: An SME Credit Committee — When To Refuse A Split


In [ ]:
# SCENARIO (illustrative): a Singapore bank's SME credit committee uses a
# boosted model to pre-rank loan applications. A committee member asks:
# "Young companies are risky — why didn't the model split on
# 'company age ≤ 2 years'?"
#
# The gain formula answers it. Take the same node as above (100 defaults,
# 800 non-defaults, current prediction 0.12) and compare the strong split
# from Task 3 with a split that isolates a SMALL segment of 40 young
# companies, 8 of which defaulted (20% vs 12% overall).

g_young = 8 * g_default + 32 * g_no_default
h_young = 40 * h_per_sample
g_rest = 92 * g_default + 768 * g_no_default
h_rest = 860 * h_per_sample
gain_young = xgb_split_gain(g_young, h_young, g_rest, h_rest, lambda_reg=1.0, gamma=0.0)

print("\n" + "=" * 70)
print("  Small-segment split vs strong split (λ = 1)")
print("=" * 70)
print(f"  Strong split (Task 3)          gain before γ: {gain:.4f}")
print(f"  'company age ≤ 2y' (40 rows)   gain before γ: {gain_young:.4f}")
for gam in [0.0, 1.0, round(gain_young * 1.5, 2)]:
    verdict = (
        "accept"
        if xgb_split_gain(g_young, h_young, g_rest, h_rest, 1.0, gam) > 0
        else "PRUNE"
    )
    print(f"    with γ = {gam:<6} → young-company split: {verdict}")
print(
    f"\n  Any γ above {gain_young:.4f} refuses the young-company split while the "
    f"strong split (gain {gain:.4f}) survives γ values up to {gain:.1f}."
)



### Checkpoint 3


In [ ]:
assert 0 < gain_young < gain, "The small-segment split must earn far less gain"
print("\n[ok] Checkpoint 3 passed — γ separates weak splits from strong ones\n")

# INTERPRETATION: The committee's intuition ("young companies are riskier")
# is correct — the segment's default rate is higher — but with only 40
# rows the loss reduction is small and the leaf weight G/(H+λ) is shrunk
# hard by λ because H is small. γ sets the minimum improvement a split
# must buy. Tuned on held-out data (never on the test set), γ is in effect
# a policy decision: higher γ = refuse to carve out small segments =
# lower training fit but more stable out-of-sample behaviour.



## REFLECTION


[x] Explained boosting as sequential residual fitting (bias reduction)
  [x] Implemented a from-scratch gradient booster with shallow trees
  [x] Watched MSE of residuals shrink round by round (the proof)
  [x] Derived the XGBoost split-gain formula from the 2nd-order Taylor
      expansion of log-loss
  [x] Interpreted λ and γ as structural regularisers that prevent the
      tree from memorising small, high-variance segments
  [x] Ran AdaBoost by hand: re-weighting puts half the weight on the
      rows the last stump missed
  [x] Connected γ to a credit committee's decision to refuse splits on
      small segments, with the gain computed rather than asserted

  KEY INSIGHT: Boosting is just gradient descent in function space. Every
  round, a new tree is the negative gradient direction in a space of
  shallow trees. The gain formula is what makes XGBoost competitive — it
  lets the tree justify every split against a complexity cost, turning
  "build a big tree" into "build only the splits that pay for themselves".

  Next: 02_xgboost.py screens the Singapore credit data for leakage,
  trains the full XGBoost classifier and ranks its feature importances.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

